# SD3.5 inpaint-EDIT — inference only (use the trained adapter)

Loads the trained edit adapter (LoRA + `input_adapter.pt`) and adds an object
into YOUR image inside a mask, preserving 100% of the background (hard-restore).
**No training** — loads the pipeline once, so no train+reload RAM OOM.

You provide, per image: a **source** image + a **mask** (white = where to add the
person, black = keep untouched). Both are resized to 512.

Requires: GPU, SD3.5 access (HF_TOKEN secret or mounted model), and the trained
adapter (mount the `run_001` you trained, e.g. as a Kaggle dataset).

## 1. Setup (pinned deps, batch-safe — no restart)

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import transformers, diffusers, torch
assert transformers.__version__ == '4.46.3', f'transformers={transformers.__version__}, expected 4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
assert torch.cuda.is_available(), 'No GPU'
print('OK on', torch.cuda.get_device_name(0))

## 2. SD3.5 access (gated)

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local); print('local SD3.5 mount')
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'Need HF_TOKEN secret or a mounted SD3.5 dataset'
    from huggingface_hub import login; login(token=HF_TOKEN); print('HF login OK')

## 3. Locate the trained adapter (flexible)

Set `ADAPTER_DIR` by hand, or let it auto-find. It must contain
`pytorch_lora_weights.safetensors` + `input_adapter.pt` (and ideally
`training_provenance.json` one level up).

In [ ]:
from pathlib import Path
ADAPTER_DIR = None   # <- set explicitly if you know it, else auto-find below

def _autofind():
    cands = []
    # mounted datasets
    for base in Path('/kaggle/input').glob('*'):
        cands += list(base.rglob('input_adapter.pt'))
    # this session's training output
    cands += list(Path('/kaggle/working/vin_lora/models').rglob('input_adapter.pt'))
    cands = [p.parent for p in cands]
    return cands[0] if cands else None

adir = Path(ADAPTER_DIR) if ADAPTER_DIR else _autofind()
assert adir is not None, ('No adapter found. Upload your run_001 as a Kaggle dataset '
                          '(must contain adapter/input_adapter.pt) and mount it, or set ADAPTER_DIR.')
assert (adir/'pytorch_lora_weights.safetensors').exists(), f'LoRA missing in {adir}'
assert (adir/'input_adapter.pt').exists(), f'input_adapter.pt missing in {adir}'
ADAPTER_DIR = adir
# run dir is the parent of adapter/ (where training_provenance.json lives)
RUN_DIR = adir.parent if adir.name == 'adapter' else adir
print('adapter dir:', ADAPTER_DIR)
print('run dir    :', RUN_DIR)

## 4. Load the edit runner (once)

In [ ]:
from LoRA.inference.sd35_edit_runner import SD35EditRunner
runner = SD35EditRunner(SD35_MODEL, ADAPTER_DIR, hf_token=HF_TOKEN).load()
print('runner ready')

## 5. Inputs — pull source images from a dataset (sources.yaml)

By default this reads real images from a mounted dataset listed in
`LoRA/configs/sources.yaml` (`SRC_SOURCE`). Provide a mask per image: either a
mask file you upload, or let it auto-make one from a random constrained
placement. Edit `SRC_SOURCE` / `N_ITEMS` / `PROMPT` below.

In [ ]:
from pathlib import Path
from LoRA.data.list_images import list_source_images

# Source images come from a dataset in sources.yaml (no manual upload).
SRC_SOURCE = 'citypersons'   # 'citypersons' | 'mot17_02' | 'human_detection'
N_ITEMS = 8
PROMPT = 'a photo of <vin_ped> pedestrian, a person standing, natural lighting'
SOURCES_YAML = Path('/kaggle/working/VIN/LoRA/configs/sources.yaml')

# Mask mode:
#   'auto'  -> random constrained placement mask (no mask files needed)
#   'files' -> you provide a mask file per image (see MASK_DIR below)
MASK_MODE = 'auto'
MASK_DIR = Path('/kaggle/working/my_masks')   # used only if MASK_MODE == 'files'

src_paths = list_source_images(SRC_SOURCE, limit=N_ITEMS, sources_path=SOURCES_YAML)
assert src_paths, f'No images for source "{SRC_SOURCE}" — mount it via Add Data.'
print(len(src_paths), 'source images from', SRC_SOURCE)

## 6. Run the edit (background preserved 100%)

In [ ]:
from PIL import Image
from LoRA.inference.random_skeleton import random_placement, PlacementConfig
OUT = Path('/kaggle/working/edit_outputs'); OUT.mkdir(parents=True, exist_ok=True)
SEED = 42; STEPS = 30
CFG = PlacementConfig(min_people=1, max_people=2)
runner.precompute_embeds([PROMPT])

def get_mask(src_img, idx, stem):
    if MASK_MODE == 'files':
        mp = MASK_DIR / f'{stem}.png'
        assert mp.exists(), f'mask file missing: {mp}'
        return Image.open(mp)
    mask, _, _ = random_placement(src_img.size, seed=SEED, index=idx, cfg=CFG)  # auto
    return mask

results = []
for i, sp in enumerate(src_paths):
    src = Image.open(sp).convert('RGB')
    msk = get_mask(src, i, sp.stem)
    out = runner.edit(src, msk, PROMPT, seed=SEED, num_inference_steps=STEPS)
    p = OUT/f'edit_{i:02d}.png'; out.save(p)
    results.append((src, msk, out, sp.stem))
    print('saved', p)
print('done ->', OUT)

## 7. Show results (source | mask | edit)

In [ ]:
from PIL import Image
from IPython.display import display
for src, msk, out, stem in results:
    cells = [src.convert('RGB'), msk.convert('RGB'), out.convert('RGB')]
    cells = [im.resize((256,256)) for im in cells]
    strip = Image.new('RGB',(768,256))
    for j,im in enumerate(cells): strip.paste(im,(256*j,0))
    print(stem, '(source | mask | edit)'); display(strip)